# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 2: *Data Merging*
##### Version Number: 3.0
---
### Contents  
> 1. *Merge sample grid with weather data*
> 2. *Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*
> 3. *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from gridMET and date from sampling grid. Sampling grid includes topological, social, inrastructure, and land cover data.
### Inputs
- Daily Weather Readings - `gridmet_final.csv` 
- Wildfire Damage Data - `fire_data.csv` (cleaned in module 1)
- California Mesh Sampling Grid - `Sampling_grids.csv` (built in appendix A) 
---
### Outputs  
- `samples_projected.csv` Cleaned weather dataset merged with fire damage severity and grid data.
---
### User Created Dependencies  

In [ ]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

---

### Data Loading and Exploration

In [3]:
weather = pd.read_csv('../data/raw/gridmet_final.csv')
fire_data = pd.read_csv("../data/processed/fire_data.csv")
samples = pd.read_csv("../data/processed/cleaned_grids.csv")

## 1. Merge sample points with weather data

#### Add geometry to sampling grid

In [ ]:
samples['geometry'] = [Point(xy) for xy in zip(samples['centroid_easting'], samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(samples, geometry='geometry', crs="EPSG:3310")

#### Filter out unused weather stations
- some station data in the weather data set are unreferenced by the sampling grid. Since merging is onto the weather data, these will produce many NA values. Only keep those stations that are in the sampling grid.

In [ ]:
sample_ids = samples['Sample_ID'].unique()
newweather = weather[weather['Sample_ID'].isin(sample_ids)]

#### Merge on Sample_ID

In [ ]:
# Merge on BOTH station and date
samples_daily = newweather.merge(
    samples_gdf,
    on=['Sample_ID'],
    how='left'
)

In [13]:
post_merge_check (samples_daily, weather)

Premerged shape:  (446340, 22)
Merged shape:  (608880, 63)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


## 2. Spatial Join of Nearest Sampling Locations with Fire Damage Datasets

To prepare for feature engineering, we spatially join each fire location to its nearest sampling grid centroid. This enables us to associate daily environmental conditions with each fire based on geographic proximity, rather than relying solely on large administrative boundaries.

#### Set geometries
- Sample grid ArcGIS work was performed in EPSG 3310 to preserve area. 

In [14]:
fire_data['geometry'] = [Point(xy) for xy in zip(fire_data['Fire_Longitude'], fire_data['Fire_Latitude'])]
fire_data_gdf = gpd.GeoDataFrame(fire_data, geometry='geometry', crs="EPSG:4326")

samples_daily_gdf = gpd.GeoDataFrame(samples_daily, geometry='geometry', crs="EPSG:3310")

## Project to Equal area projection for more accuracy
samples_projected = samples_daily_gdf.to_crs(3310)
fire_data_projected = fire_data_gdf.to_crs(3310)

- Centroids are loosely spaced 46000 metyers apart. Set buffer distance a small bit above this value. This parameter is adjustable.

In [ ]:
buffer_dist = 47163

#### Main loop of Spatial Join algorithm
- Loop through all dates in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move to next date
- Load centroids associated with the current date
- Create a buffer around each around each fire for current date
- Intersection spatial join of sampling centroids and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - Total estimated cost is summed for all fires in range
- Assign samples to new dataframe

In [16]:
## Initialize variables to track total damage and number of fires associated with each point
samples_projected['fire_count'] = 0
fire_data_projected['total_fire_damage'] = 0.0

In [17]:
for dt in fire_data_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_data_projected[fire_data_projected['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_projected[samples_projected['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each fire
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered.buffer(buffer_dist)

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    # Aggregate counts and total damage per sample
    grouped = joined.groupby('grid_id').agg(
        fire_count=('Estimated Damage', 'count'),
        total_fire_damage=('Estimated Damage', 'sum'),
        acres = ('Acres', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_projected.loc[samples_projected['Date'] == dt, 'fire_count'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'grid_id'].map(grouped['fire_count']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'total_fire_damage'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'grid_id'].map(grouped['total_fire_damage']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'acres'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'grid_id'].map(grouped['acres']).fillna(0)

In [18]:
post_merge_check(samples_projected, samples_daily)

Premerged shape:  (608880, 63)
Merged shape:  (608880, 66)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  12272


In [19]:
samples_projected.isna().sum()

Sample_ID                       0
Date                            0
Burning Index                   0
Energy Release Component        0
Actual Evapotranspiration       0
                             ... 
minimum_x                       0
geometry                        0
fire_count                      0
total_fire_damage            6136
acres                        6136
Length: 66, dtype: int64

**Note**: NA values are for buffered areas without any fires in range within the date range. Fill these values with 0. 

In [20]:
samples_projected['total_fire_damage'] = samples_projected['total_fire_damage'].fillna(0)
samples_projected['acres'] = samples_projected['acres'].fillna(0)

#### Sort dataframe by date

In [21]:
## sort dataframe by date
samples_projected = samples_projected.sort_values(by='Date')

## reset index
samples_projected = samples_projected.reset_index(drop=True)

---

## 3. Export File

In [22]:
samples_projected.to_csv('../data/processed/samples_projected.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
